# Getting the Bronze Data From External storage

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins

base_path = "abfss://orderitems@adlsstdout001tgt.dfs.core.windows.net/OrderItems/"
# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
raw_df = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(base_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
import builtins
from pyspark.sql.functions import current_date, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import col
from pyspark.sql.functions import *
from pyspark.sql.types import *

# ==========================================
# STEP 2 : REMOVE DUPLICATES
# ==========================================
# Business Key:
# One order can have multiple items
# Therefore unique combination:
# (order_id + order_item_id)

df = raw_df.dropDuplicates(["order_id", "order_item_id"])

# ==========================================
# STEP 3 : TRIM ALL STRING COLUMNS
# ==========================================

string_cols = [col for col, dtype in df.dtypes if dtype == "string"]

for c in string_cols:
    df = df.withColumn(c, F.trim(F.col(c)))

# ==========================================
# STEP 4 : STANDARDIZE COLUMN NAMES
# ==========================================

for c in df.columns:
    cleaned_name = (
        c.strip()
         .lower()
         .replace(" ", "_")
         .replace("-", "_")
    )
    df = df.withColumnRenamed(c, cleaned_name)

# ==========================================
# STEP 5 : CLEAN ID COLUMNS
# ==========================================
# Remove unwanted special characters
# Keep only alphanumeric

id_columns = [
    "order_id",
    "product_id",
    "seller_id"
]

for c in id_columns:
    df = df.withColumn(
        c,
        F.regexp_replace(F.col(c), "[^A-Za-z0-9]", "")
    )

# ==========================================
# STEP 6 : HANDLE NULLS
# ==========================================

df = df.na.drop(
    subset=[
        "order_id",
        "order_item_id",
        "product_id",
        "seller_id"
    ]
)

# ==========================================
# STEP 7 : DATA TYPE CASTING
# ==========================================

df = df \
    .withColumn(
        "order_item_id",
        F.col("order_item_id").cast(IntegerType())
    ) \
    .withColumn(
        "price",
        F.round(F.col("price").cast(DecimalType(10,2)), 2)
    ) \
    .withColumn(
        "freight_value",
        F.round(F.col("freight_value").cast(DecimalType(10,2)), 2)
    ) \
    .withColumn(
        "shipping_limit_date",
        F.to_timestamp(
            F.col("shipping_limit_date"),
            "yyyy-MM-dd HH:mm:ss"
        )
    ) \
    .withColumn(
        "ingest_date",
        F.to_date("ingest_date")
    ) \
    .withColumn(
        "file_date",
        F.to_date("file_date")
    )

# ==========================================
# STEP 8 : BUSINESS RULE VALIDATIONS
# ==========================================

# Price cannot be negative
df = df.filter(F.col("price") >= 0)

# Freight cannot be negative
df = df.filter(F.col("freight_value") >= 0)

# Order item id should be positive
df = df.filter(F.col("order_item_id") > 0)

# ==========================================
# STEP 9 : HANDLE INVALID DATES
# ==========================================

df = df.filter(
    F.col("shipping_limit_date").isNotNull()
)

# ==========================================
# STEP 10 : ADD AUDIT COLUMNS
# ==========================================

df = df \
    .withColumn(
        "created_timestamp",
        F.current_timestamp()
    ) \
    .withColumn(
        "updated_timestamp",
        F.current_timestamp()
    ) \
    .withColumn(
        "source_system",
        F.lit("ecommerce")
    )

# ==========================================
# STEP 11 : DATA QUALITY FLAGS
# ==========================================

df = df.withColumn(
    "dq_status",
    F.when(
        (F.col("price") > 0) &
        (F.col("freight_value") >= 0),
        "VALID"
    ).otherwise("INVALID")
)

# ==========================================
# STEP 12 : REORDER COLUMNS
# ==========================================

final_df = df.select(
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "ingest_date",
    "file_date",
    "dq_status",
    "source_system",
    "created_timestamp",
    "updated_timestamp"
)

# ==========================================
# STEP 15 : DATA VALIDATION CHECKS
# ==========================================

print("Total Records :", final_df.count())

print("Duplicate Records :",
      final_df.count() -
      final_df.dropDuplicates(
          ["order_id", "order_item_id"]
      ).count()
)

final_df.printSchema()

display(final_df.limit(10))

In [0]:
tablename = "ecom.silver.fact_silver_orderitems"

In [0]:
(
        final_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
            "path",
            "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Orderitems"
        )
        .saveAsTable(tablename)
    )

print("Full Load Completed Successfully")